In [1]:
import nibabel as nib
import os
import glob
import numpy as np
from pathlib import Path

def extract_regional_activation(activation_nifti_path, aal_atlas_path, output_path=None):
    """
    Extract mean activation values for each AAL brain region.

    Parameters:
    -----------
    activation_nifti_path : str
        Path to the brain activation map (NIfTI file)
    aal_atlas_path : str
        Path to the AAL atlas (NIfTI file)
    output_path : str, optional
        Path to save the output vector (as .npy file)

    Returns:
    --------
    regional_activation : numpy array
        Vector of shape (num_regions, 1) with mean activation per region
    region_labels : numpy array
        Array of region label IDs corresponding to each value
    """

    # Load the NIfTI files
    print("Loading activation map...")
    activation_img = nib.load(activation_nifti_path)
    activation_data = activation_img.get_fdata()

    print("Loading AAL atlas...")
    atlas_img = nib.load(aal_atlas_path)
    atlas_data = atlas_img.get_fdata()

    # Check that dimensions match
    if activation_data.shape != atlas_data.shape:
        raise ValueError(f"Shape mismatch: activation {activation_data.shape} vs atlas {atlas_data.shape}")

    # Get unique region labels (excluding 0, which is background)
    region_labels = np.unique(atlas_data)
    region_labels = region_labels[region_labels != 0]  # Remove background

    print(f"Found {len(region_labels)} brain regions")

    # Calculate mean activation for each region
    regional_activation = np.zeros((len(region_labels), 1))

    for idx, region_label in enumerate(region_labels):
        # Create mask for current region
        region_mask = (atlas_data == region_label)

        # Extract activation values for this region
        region_activation_values = activation_data[region_mask]

        # Calculate mean (ignoring NaN values if present)
        regional_activation[idx, 0] = np.nanmean(region_activation_values)

    print(f"Output shape: {regional_activation.shape}")

    # Save if output path is provided
    if output_path:
        np.save(output_path, regional_activation)
        print(f"Saved to {output_path}")

    return regional_activation, region_labels




In [ ]:
# # Example usage
# if __name__ == "__main__":
#     # Replace these paths with your actual file paths
#     activation_path = "examples/language_association-test_z_FDR_0.01.nii.gz"
#     atlas_path = "/Users/huilisun/Library/CloudStorage/OneDrive-PennO365/Atlas/AAL2/AAL2_mni.nii.gz"
#
#     # Extract regional activation
#     activation_vector, regions = extract_regional_activation(
#         activation_path,
#         atlas_path,
#         output_path="examples/language_regional_activation.npy"
#     )
#
#     # Display results
#     print("\nRegional Activation Summary:")
#     print(f"Shape: {activation_vector.shape}")
#     print(f"Mean activation across regions: {np.mean(activation_vector):.4f}")
#     print(f"Min activation: {np.min(activation_vector):.4f}")
#     print(f"Max activation: {np.max(activation_vector):.4f}")
#
#     print(activation_vector)
#


In [2]:
# Example usage
if __name__ == "__main__":
    # Define paths
    atlas_path = "/Users/huilisun/Library/CloudStorage/OneDrive-PennO365/Atlas/AAL2/AAL2_mni.nii.gz"
    input_dir = "/Users/huilisun/Library/CloudStorage/OneDrive-PennO365/ControlCog/Activation/Neurosynth/data/ALL"  # Directory containing your activation files
    output_dir = "/Users/huilisun/Library/CloudStorage/OneDrive-PennO365/ControlCog/Activation/Neurosynth/data/association"

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Find all matching files
    pattern = os.path.join(input_dir, "*association-test_z_FDR_0.01.nii.gz")
    activation_files = glob.glob(pattern)

    print(f"Found {len(activation_files)} files to process\n")

    # Process each file
    for activation_path in activation_files:
        # Extract the base name for output file
        base_name = Path(activation_path).stem  # Removes .gz
        base_name = Path(base_name).stem
        base_name = base_name.replace("association-test_z_FDR_0.01", "regional_activation")
        output_path = os.path.join(output_dir, f"{base_name}.npy")

        print(f"Processing: {os.path.basename(activation_path)}")

        try:
            # Extract regional activation
            activation_vector, regions = extract_regional_activation(
                activation_path,
                atlas_path,
                output_path=output_path
            )

            # Display results
            print(f"  Shape: {activation_vector.shape}")
            print(f"  Mean activation: {np.mean(activation_vector):.4f}")
            print(f"  Min activation: {np.min(activation_vector):.4f}")
            print(f"  Max activation: {np.max(activation_vector):.4f}")
            print(f"  Saved to: {output_path}\n")

        except Exception as e:
            print(f"  Error processing {activation_path}: {str(e)}\n")

    print("Processing complete!")

Found 100 files to process

Processing: music_association-test_z_FDR_0.01.nii.gz
Loading activation map...
Loading AAL atlas...
Found 120 brain regions
Output shape: (120, 1)
Saved to /Users/huilisun/Library/CloudStorage/OneDrive-PennO365/ControlCog/Activation/Neurosynth/data/association/music_regional_activation.npy
  Shape: (120, 1)
  Mean activation: 0.1227
  Min activation: 0.0000
  Max activation: 3.4324
  Saved to: /Users/huilisun/Library/CloudStorage/OneDrive-PennO365/ControlCog/Activation/Neurosynth/data/association/music_regional_activation.npy

Processing: communication_association-test_z_FDR_0.01.nii.gz
Loading activation map...
Loading AAL atlas...
Found 120 brain regions
Output shape: (120, 1)
Saved to /Users/huilisun/Library/CloudStorage/OneDrive-PennO365/ControlCog/Activation/Neurosynth/data/association/communication_regional_activation.npy
  Shape: (120, 1)
  Mean activation: 0.0103
  Min activation: 0.0000
  Max activation: 0.3580
  Saved to: /Users/huilisun/Library/Cl

read all activations and save in csv files with column as activation names and rows as nodes

In [6]:
action = np.load('association/action_regional_activation.npy')
action_binary =  (action > 0).astype(int)


In [4]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Define the directory containing activation files
activation_dir = 'association'

# Dictionaries to store all activations
activations = {}
activations_binary = {}

# Automatically find all files matching the pattern *_regional_activation.npy
activation_files = [f for f in os.listdir(activation_dir)
                   if f.endswith('_regional_activation.npy')]

print(f"Found {len(activation_files)} activation files:")
for f in sorted(activation_files):
    print(f"  - {f}")

# Read each activation file
for filename in activation_files:
    filepath = os.path.join(activation_dir, filename)

    try:
        # Load the numpy array
        data = np.load(filepath)

        # Extract activation name from filename (remove path and extension)
        activation_name = Path(filename).stem.replace('_regional_activation', '')

        # Flatten the array if it's 120x1 to just 120 elements
        data_flat = data.flatten()

        # Create binary version (1 if > 0, else 0)
        data_binary = (data_flat > 0).astype(int)

        # Store in dictionaries
        activations[activation_name] = data_flat
        activations_binary[activation_name] = data_binary

        print(f"Loaded {activation_name}: shape {data.shape} -> {data_flat.shape}")

    except FileNotFoundError:
        print(f"Warning: File not found - {filepath}")
    except Exception as e:
        print(f"Error loading {filename}: {str(e)}")

# Create DataFrames with activations as columns and nodes as rows
if activations:
    # Original activations
    df = pd.DataFrame(activations)
    df.insert(0, 'node', range(len(df)))

    # Binary activations
    df_binary = pd.DataFrame(activations_binary)
    df_binary.insert(0, 'node', range(len(df_binary)))

    # Save to CSV files
    output_file = 'regional_activations.csv'
    output_file_binary = 'regional_activations_binary.csv'

    df.to_csv(output_file, index=False)
    df_binary.to_csv(output_file_binary, index=False)

    print(f"\nSuccessfully saved {len(df)} nodes and {len(activations)} activations")
    print(f"  - Original: {output_file}")
    print(f"  - Binary: {output_file_binary}")
    print(f"Shape: {df.shape}")
    print(f"\nFirst few rows (original):")
    print(df.head())
    print(f"\nFirst few rows (binary):")
    print(df_binary.head())
else:
    print("No activation data was loaded.")

Found 100 activation files:
  - action_regional_activation.npy
  - anxiety_regional_activation.npy
  - arousal_regional_activation.npy
  - association_regional_activation.npy
  - attention_regional_activation.npy
  - auditory_regional_activation.npy
  - categorization_regional_activation.npy
  - cognitive_regional_activation.npy
  - communication_regional_activation.npy
  - comprehension_regional_activation.npy
  - concept_regional_activation.npy
  - conflict_regional_activation.npy
  - context_regional_activation.npy
  - control_regional_activation.npy
  - decision making_regional_activation.npy
  - decision_regional_activation.npy
  - discrimination_regional_activation.npy
  - distraction_regional_activation.npy
  - effort_regional_activation.npy
  - emotion regulation_regional_activation.npy
  - emotion_regional_activation.npy
  - empathy_regional_activation.npy
  - expression_regional_activation.npy
  - face recognition_regional_activation.npy
  - face_regional_activation.npy
  - f